In [ ]:
!pip -q install SimpleITK

from google.colab import drive
import os, json, glob
from pathlib import Path
import numpy as np, pandas as pd, cv2, SimpleITK as sitk, torch, torchvision
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

NODE21  = Path('/content/drive/MyDrive/Algoverse/data/node21')
MHA_DIR = NODE21/'images'
ANN_CSV = NODE21/'metadata.csv'
CKPT    = Path('/content/drive/MyDrive/Algoverse/Teammates/baseline1_checkpoint.pth')
OUT     = Path('/content/baselines'); OUT.mkdir(exist_ok=True)
for p in (MHA_DIR, ANN_CSV, CKPT): assert p.exists(), f'{p} missing'

SIZE, DET_SIZE, SCORE_MIN, CHEST_MM = 512, 800, 0.05, 350.0
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

DET = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None,
    box_score_thresh=0.0,           # 0.0 -- F6: two stacked filters at 0.05 truncated
    box_detections_per_img=300)     # the old FROC curve into five duplicate points
DET.roi_heads.box_predictor = FastRCNNPredictor(
    DET.roi_heads.box_predictor.cls_score.in_features, 2)
_st = torch.load(CKPT, map_location=DEV, weights_only=False)
DET.load_state_dict(_st['model'] if isinstance(_st, dict) and 'model' in _st else _st)
DET = DET.eval().to(DEV)
print(f'detector on {DEV}, checkpoint {CKPT.stat().st_size/1e6:.0f} MB')
print('internal transform min_size =', DET.transform.min_size)

In [ ]:
def load_chest(path, size=SIZE):
    """IDENTICAL to the generation pipeline. Real and synthetic must share this or any
    comparison confounds resolution with lesion type."""
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    oh, ow = a.shape
    s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
    a = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32)/255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB'), (ow, oh, s, x0, y0)

@torch.no_grad()
def detect_all(pil, size=DET_SIZE):
    im = pil.convert('RGB').resize((size,size), Image.LANCZOS)
    o = DET([TF.to_tensor(im).to(DEV)])[0]
    return o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()

def centre_in(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]

In [ ]:
INCLUDE_NEGATIVES = True     # needed for a real false-positive rate

raw = pd.read_csv(ANN_CSV)
raw = raw[raw.img_name != 'n0507.mha']          # duplicate of n1059, different annotations
gt = {n: g for n, g in raw[raw.label == 1].groupby('img_name')}
names = sorted(gt) + (sorted(set(raw[raw.label == 0].img_name) - set(gt))
                      if INCLUDE_NEGATIVES else [])
print(f'{len(names)} images ({len(gt)} with nodules)')

(OUT/'dets').mkdir(exist_ok=True)
rows, nod = [], []
for i, name in enumerate(names):
    p = MHA_DIR/name
    if not p.exists(): continue
    img, (W,H,s,x0,y0) = load_chest(p)
    b, sc = detect_all(img)
    stem = name.replace('.mha','')
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist()},
              open(OUT/'dets'/f'{stem}.json','w'))    # raw, so no threshold needs a rerun

    targets = []
    for j, r in enumerate(gt.get(name, pd.DataFrame()).itertuples()):
        fx0, fy0 = (r.x-x0)/s, (r.y-y0)/s
        fx1, fy1 = (r.x+r.width-x0)/s, (r.y+r.height-y0)/s
        if not (0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1):
            continue                                  # drop, never clamp
        t = (fx0, fy0, fx1, fy1); targets.append(t)
        hits = [ss for bb, ss in zip(b, sc) if centre_in(bb, t)]
        nod.append(dict(img_name=stem, nodule=j, fx0=fx0, fy0=fy0, fx1=fx1, fy1=fy1,
                        approx_mm=round(CHEST_MM*(fx1-fx0),1),
                        best_score=round(float(max(hits, default=0.0)), 4)))
    rows.append(dict(img_name=stem, n_gt=len(targets), n_boxes=int(len(sc)),
                     max_score=round(float(sc.max()) if len(sc) else 0.0, 4)))
    if i % 200 == 0: print(f'  {i}/{len(names)}')

img_df = pd.DataFrame(rows);  nod_df = pd.DataFrame(nod)
img_df.to_csv(OUT/'per_image.csv', index=False)
nod_df.to_csv(OUT/'per_nodule.csv', index=False)
print(f'\n{len(img_df)} images, {len(nod_df)} nodules scored')
print(nod_df.best_score.describe().round(3).to_string())

In [ ]:
def froc(nod_df, img_df, thresholds=np.unique(np.concatenate(
        [np.linspace(0,1,201), nod_df.best_score.values]))):
    n_img, n_nod = len(img_df), len(nod_df)
    pts = []
    for t in thresholds:
        tp = int((nod_df.best_score >= t).sum())
        fp = 0
        for _, r in img_df.iterrows():
            d = json.load(open(OUT/'dets'/f'{r.img_name}.json'))
            sc = np.array(d['scores']); b = np.array(d['boxes']).reshape(-1,4)
            keep = sc >= t
            g = nod_df[nod_df.img_name == r.img_name]
            tgts = [(x.fx0,x.fy0,x.fx1,x.fy1) for x in g.itertuples()]
            fp += sum(1 for bb in b[keep] if not any(centre_in(bb, t_) for t_ in tgts))
        pts.append((t, tp/n_nod, fp/n_img))
    return pd.DataFrame(pts, columns=['threshold','sensitivity','fp_per_image'])

F = froc(nod_df, img_df)
F.to_csv(OUT/'froc.csv', index=False)

OPS = [0.125, 0.25, 0.5, 1, 2, 4, 8]
tab = []
for op in OPS:
    sub = F[F.fp_per_image <= op]
    tab.append(dict(fp_per_image=op,
                    sensitivity=round(sub.sensitivity.max(), 4) if len(sub) else np.nan))
T = pd.DataFrame(tab)
print(T.to_string(index=False))
print(f'\nFROC score (mean of the 7 operating points): {T.sensitivity.mean():.4f}')
print(f'distinct sensitivity values: {T.sensitivity.nunique()} of 7')
print('  -> F6: the old curve had 5 of 7 identical because two filters at 0.05 were'
      '\n     stacked. If this still shows duplicates, the detector genuinely runs out'
      '\n     of boxes rather than being truncated by a threshold.')
T.to_csv(OUT/'table-froc.csv', index=False)

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p, d = k/n, 1 + z**2/n
    c = (p + z**2/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))/d
    return max(0,c-h), min(1,c+h)

# abstain on an IMAGE when the detector's top score sits in an uncertain band
merged = nod_df.merge(img_df[['img_name','max_score']], on='img_name')
rows = []
for a in np.arange(0.0, 0.95, 0.05):
    abst = merged.max_score < a                    # abstain: too unconfident to answer
    ans  = merged[~abst]
    if len(ans) == 0: continue
    fn = (ans.best_score < SCORE_MIN).sum()
    lo, hi = wilson(fn, len(ans))
    rows.append(dict(abstain_below=round(a,2),
                     abstention_rate=round(abst.mean(), 4),
                     n_answered=len(ans),
                     fnr_answered=round(fn/len(ans), 4),
                     fnr_lo=round(lo,4), fnr_hi=round(hi,4)))
B2 = pd.DataFrame(rows)
B2.to_csv(OUT/'table-baseline2-abstention.csv', index=False)
print(B2.to_string(index=False))
print('\nEvery rate carries a Wilson interval (reviewer item 10 -- Table 1 had none).')
print('Wilson rather than normal because these sit near 0 and 1 where the normal'
      '\ninterval misbehaves.')

In [ ]:
INK,EDGE,GRIDC,BLUE,SLATE,CORAL = '#2c3e50','#1f2d4d','#eeeeee','#6694de','#7982a6','#fc7c7c'
plt.rcParams.update({'savefig.dpi':300,'figure.facecolor':'white','font.size':12,
    'axes.titleweight':'bold','axes.titlecolor':INK,'axes.labelweight':'bold',
    'axes.labelcolor':INK,'text.color':INK,'xtick.color':INK,'ytick.color':INK,
    'axes.edgecolor':INK,'axes.spines.top':False,'axes.spines.right':False,
    'grid.color':GRIDC,'axes.axisbelow':True})

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
f = F[F.fp_per_image > 0].sort_values('fp_per_image')
ax[0].semilogx(f.fp_per_image, f.sensitivity, color=BLUE, lw=2.4)
ax[0].scatter(T.fp_per_image, T.sensitivity, color=BLUE_DEEP if False else '#3c5182',
              zorder=5, s=40, edgecolor=EDGE)
ax[0].set_xlabel('False positives per image'); ax[0].set_ylabel('Sensitivity')
ax[0].set_title(f'FROC — mean {T.sensitivity.mean():.3f}')
ax[0].set_ylim(0,1.02); ax[0].grid(True, which='both', color=GRIDC)

ax[1].plot(B2.abstention_rate, B2.fnr_answered, color=SLATE, lw=2.4, marker='o', ms=4)
ax[1].fill_between(B2.abstention_rate, B2.fnr_lo, B2.fnr_hi, color=SLATE, alpha=.2)
ax[1].set_xlabel('Abstention rate'); ax[1].set_ylabel('FNR on answered cases')
ax[1].set_title('Baseline 2 — confidence-only abstention')
ax[1].grid(True, color=GRIDC)
plt.suptitle('Baselines on NODE21', fontsize=17, fontweight='bold', color=INK)
plt.tight_layout(rect=[0,0,1,0.93])
for ext in ('png','pdf'):
    fig.savefig(OUT/f'fig-baselines.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import shutil
DEST = Path('/content/drive/MyDrive/Algoverse/results'); DEST.mkdir(parents=True, exist_ok=True)
z = shutil.make_archive('/content/baselines', 'zip', OUT)
shutil.copy(z, DEST/'baselines.zip')
print(f'{Path(z).stat().st_size/1e6:.1f} MB -> {DEST}  [safe]')
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print(f'download skipped ({e}) -- Drive copy is intact')

In [ ]:
print("life is like a box of chocolates...")